# The normal brain across cohorts

What the cohorts' brain-state dynamics **share**, more than how they differ. One state set (the
same atlas × window × state definition for everyone), every cohort's rest and movie scans, all
subjects (no phenotype split: these are the cohorts as they are).

* **Train and projected.** The states are fitted on the training cohorts and every other cohort is
  labelled by that fit, never seen by it (`cluster.py`, `DEFAULT_TRAIN`). A projected cohort that
  looks like the training ones is evidence that the states generalise.
* **Equivalence, not "no difference".** A non-significant difference does not show that two cohorts
  agree. Figures 2 and 4 use a TOST (two one-sided tests, Lakens 2017): a cohort is **equivalent** to
  the others on a variable when the 90% CI of its difference lies inside ± SESOI, the smallest
  difference that would matter. SESOI = `sesoi_d` × the pooled SD of that variable; 0.2 is Cohen's
  "small effect", a convention to justify (or replace by a bound in real units). A small cohort has a
  wide CI and may not be shown equivalent: that is not evidence of a difference.

The tables come from `cohort_tables.py` and the figures from `cohort_plots.py`; the state diagram is
notebook 10's, with the same label rules. Nothing here writes to `outputs/`.

In [ ]:
import cohort_tables as ct, cohort_plots as cp

cfg = ct.Settings(
    space=dict(atlas="yeo7", window_s=30.0, states="HMM1_pca3_8"),   # see the list printed below
    # label -> the cohort= directories it reads, and its role in the state fit. A new cohort is one line.
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected")},
    rest_pattern=r"rest",          # task= values matching this are rest; every other task is movie
    # figure 1: movie vs rest
    alpha=0.10,                    # BH-FDR
    fc_task=1.0,                   # |log2FC| for the graph (1 = twofold)
    fc_dyn=0.25,                   # |log2FC| for the dynamics
    edge_top=12,                   # the mean's strongest arrows, per task
    # figures 2 and 4: equivalence
    sesoi_d=0.2,                   # TOST bounds, in pooled SDs
    top_transitions=8,             # figure 2: the strongest transitions shown
    save_dir=None,                 # e.g. "figures/11" to save PNG + SVG
)
sets = ct.available_sets(cfg.root, cfg.cohorts)
print(f"{len(sets)} state sets have every cohort; e.g.")
for _, r in sets.head(8).iterrows():
    print(f"  space=dict(atlas={r.atlas!r}, window_s={r.window_s}, states={r.states!r})")
co = ct.Cohorts(cfg)
co.counts()

## 1. Rest and movie, per cohort and together

Rows are the cohorts, then their mean (every cohort weighs once); columns are rest and movie. Each
diagram is the cohort's own dynamics, averaged over its subjects (a subject's runs or films first).
**Circle area** is time in a state, **the loop** is p(stay), **arrow width** is p(j | i), on one
width scale for the whole figure; every diagram draws the mean's `edge_top` strongest arrows of each
task. **Blue** on the movie is what differs from the **same people's** rest: paired t-test (Welch if
a cohort's rest and movie subjects do not overlap), BH-FDR over every variable, and |log2FC| ≥
`fc_task` (`fc_dyn` for the dynamics), with the change in pp. Beside each diagram are the dynamics:
⇄ switch rate, stopwatch = mean dwell, H entropy rate, ✳ dispersion, # distinct transitions.

In [ ]:
graphs = co.graphs()
marks, dyn = co.task_change()
cp.fig_diagrams(co, graphs, marks, dyn)
# marks[marks["hit"]].sort_values(["cohort", "p"])

## 2. Is every cohort equivalent to the others?

Movie only (the task every cohort shares). For each variable and cohort: the cohort's mean minus the
mean of all the others, with its 90% CI, in the variable's own units (pp for the probabilities).
The grey band is ± SESOI. **Filled** = equivalent (the CI is inside the band), **hollow** = not
shown equivalent; **✓** at the right when every cohort is. The transitions are the
`top_transitions` strongest; every occupancy and every dynamics variable is shown, the dynamics each
on its own axis.

In [ ]:
agree = co.agreement()
cp.fig_agreement(co, agree)
# agree.groupby("variable")["equivalent"].all().value_counts()

## 3. Does each projected cohort reproduce the training cohorts?

Movie, one row per projected cohort. Every variable's mean in the projected cohort (y) against its
mean in each training cohort (x, a dot per training cohort, in its colour): on the diagonal, the
projection reproduces the training. The title gives the correlation with each training cohort and
between the training cohorts: a projected cohort that matches them about as well as they match each
other is as close as two training cohorts are. The dynamics are log-log (they span 0.4 to ~35).

In [ ]:
means = co.cohort_means()
cp.fig_train_vs_projected(co, means)
# means.T

## 4. The transitions as matrices

Movie p(j | i) per cohort, on one colour scale, then figure 2's TOST for every cell: **✓** where
every cohort is equivalent to the others, a **red frame** with k/N where only k of the N cohorts
are. The grey is the largest |cohort − others| reached by a 90% CI end, in pp.

In [ ]:
cp.fig_matrix(co, graphs, agree)